In [1]:
# -*- coding: utf-8 -*-
"""
颗粒相ClPAHs预测（2015 / 2017 / 2019 / 2021 / 2023，全国168城×12月）
------------------------------------------------------------
功能（合并原 notebook 中 3 个重复预测 cell 后的唯一完整版）：
1. 用训练数据重建最终颗粒相模型（固定最终参数）
2. 评估模型性能：RepeatedKFold OOF + 随机留一holdout(bagging)，输出 R2/log_R2/MAE/RMSE
   并保存 model_performance_metrics.csv 与 预测说明.json
3. 预测 2015 / 2017 / 2019 / 2021 / 2023 三个文件中的每一行ClPAHs浓度
4. 用 RepeatedKFold OOF 残差估计 log1p 空间不确定性，做 Monte Carlo (MC)
5. 每一行输出：点预测 + MC 均值 / 分位数 / CI宽度 / IQR（仿气相格式）
6. 同时输出逐行点预测结果（pred_concentration）与月度分位数统计
7. 合并输出所有年份的月度分位数总表

说明：
- 预测文件无城市列，但共有2016行 = 168城 × 12月
- 行顺序判断为：每个城市连续12个月
- 因此 month = ((行号-1) % 12) + 1
"""

import os
import gc
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import RepeatedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# =========================
# PATH CONFIG
# =========================
TRAIN_FILE_WINDOWS = r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\20260311颗粒相ClPAHs建模-3.csv"
TRAIN_FILE_FALLBACK = r"/mnt/data/20260311颗粒相ClPAHs建模-3.csv"

PRED_FILES = {
    "2015": r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2015年预测.csv",
    "2017": r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2017年预测.csv",
    "2019": r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2019年预测.csv",
    "2021": r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2021年预测.csv",
    "2023": r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2023年预测.csv",
}

PRED_FILES_FALLBACK = {
    "2015": r"/mnt/data/2015年预测.csv",
    "2017": r"/mnt/data/2017年预测.csv",
    "2019": r"/mnt/data/2019年预测.csv",
    "2021": r"/mnt/data/2021年预测.csv",
    "2023": r"/mnt/data/2023年预测.csv",
}

OUT_DIR = r"C:\Users\HAUWEI\XGBoost建模\颗粒相ClPAHs_全年份预测_MC"

# 预测年份按时间顺序
YEAR_ORDER = ["2015", "2017", "2019", "2021", "2023"]


# =========================
# MODEL CONFIG
# =========================
RANDOM_STATE = 42
NTHREAD = 4

MISSING_RATE_THRESHOLD = 0.60
WINSORIZE_Q_LOW = 0.01
WINSORIZE_Q_HIGH = 0.99
CORR_THRESHOLD = 0.985

NUM_BOOST_ROUND_FINAL = 6500

# 评估配置（与建模+SHAP脚本保持同一套口径：随机RepeatedCV + 随机留一holdout bagging）
TEST_SIZE = 0.20
HOLDOUT_BAG_SEEDS = [42, 52, 62, 72, 82, 92, 102, 112, 122]

# 用CV残差估计log空间不确定性
CV_SPLITS = 5
CV_REPEATS = 5

# MC设置
MC_SAMPLES = 1000

BEST_PARAMS = {
    "eta": 0.01,
    "max_depth": 5,
    "min_child_weight": 24.0,
    "subsample": 0.9,
    "colsample_bytree": 0.64,
    "gamma": 0.1,
    "alpha": 0.0,
    "lambda_": 10.0,
    "max_delta_step": 0.0
}


# =========================
# UTILS
# =========================
def ensure_outdir(p):
    os.makedirs(p, exist_ok=True)

def resolve_path(win_path, fallback_path=None):
    if os.path.exists(win_path):
        return win_path
    if fallback_path and os.path.exists(fallback_path):
        return fallback_path
    raise FileNotFoundError(win_path if fallback_path is None else f"{win_path} | {fallback_path}")

def read_csv_auto(fp):
    for enc in ["utf-8-sig", "gbk", "gb18030", "utf-8"]:
        try:
            return pd.read_csv(fp, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(fp)

def safe_num_series(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )

def metric_pack(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred, float), 0, None)
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "log_R2": float(r2_score(np.log1p(np.asarray(y_true, float)), np.log1p(y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

def make_y_bins(y, n_bins=5):
    y = pd.Series(y).reset_index(drop=True)
    n_unique = y.nunique()
    n_bins = min(n_bins, max(2, n_unique))
    try:
        bins = pd.qcut(y, q=n_bins, labels=False, duplicates="drop")
    except Exception:
        bins = pd.qcut(y.rank(method="first"), q=n_bins, labels=False, duplicates="drop")
    return bins.astype(int)


# =========================
# FEATURE ENGINEERING
# =========================
def add_quarter_features(df):
    if "quarter" not in df.columns:
        return df
    q = pd.to_numeric(df["quarter"], errors="coerce")
    df["quarter_sin"] = np.sin(2 * np.pi * q / 4.0)
    df["quarter_cos"] = np.cos(2 * np.pi * q / 4.0)
    return df

def add_interaction_features(df):
    cols = set(df.columns)

    if {"PM2.5", "BLH"}.issubset(cols):
        df["PM25_x_BLH"] = df["PM2.5"] * df["BLH"]
    if {"temperature", "humidity"}.issubset(cols):
        df["Temp_x_Humidity"] = df["temperature"] * df["humidity"]
    if {"precipitation", "PM2.5"}.issubset(cols):
        df["Prec_x_PM25"] = df["precipitation"] * df["PM2.5"]
    if {"CO", "SO2"}.issubset(cols):
        df["CO_x_SO2"] = df["CO"] * df["SO2"]
    if {"centralized_heating_or_not", "quarter"}.issubset(cols):
        df["Heating_x_Quarter"] = df["centralized_heating_or_not"] * df["quarter"]
    if {"heavy_industrial_or_not", "PM10"}.issubset(cols):
        df["Industry_x_PM10"] = df["heavy_industrial_or_not"] * df["PM10"]
    if {"PM10", "BLH"}.issubset(cols):
        df["PM10_x_BLH"] = df["PM10"] * df["BLH"]
    if {"NO2", "PM2.5"}.issubset(cols):
        df["NO2_x_PM25"] = df["NO2"] * df["PM2.5"]

    return df


# =========================
# TRAIN DATA PREP
# =========================
def prepare_training_data(df):
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all").copy()
    df.columns = [str(c).strip() for c in df.columns]

    cols_lower_map = {str(c).strip().lower(): c for c in df.columns}

    if "sample_name" not in cols_lower_map:
        raise ValueError(f"训练文件没有找到 sample_name 列，当前列名：{df.columns.tolist()}")
    sample_col = cols_lower_map["sample_name"]

    if "concentration" not in cols_lower_map:
        raise ValueError(f"训练文件没有找到 concentration 列，当前列名：{df.columns.tolist()}")
    target_col = cols_lower_map["concentration"]

    raw_feature_cols = [c for c in df.columns if c not in [sample_col, target_col]]

    work = df[[sample_col, target_col] + raw_feature_cols].copy()
    work[sample_col] = work[sample_col].astype(str).str.strip()
    work[target_col] = safe_num_series(work[target_col])

    for c in raw_feature_cols:
        work[c] = safe_num_series(work[c])

    work = add_quarter_features(work)
    work = add_interaction_features(work)

    work = work.dropna(subset=[target_col]).copy()
    work = work[work[target_col] >= 0].copy()
    work = work[work[sample_col].notna()].copy()
    work = work[work[sample_col].astype(str).str.len() > 0].copy()

    feature_cols = [c for c in work.columns if c not in [sample_col, target_col]]

    miss = work[feature_cols].isna().mean()
    feature_cols = miss[miss <= MISSING_RATE_THRESHOLD].index.tolist()
    if not feature_cols:
        raise ValueError("训练文件缺失率过滤后无特征。")

    for c in feature_cols:
        s = work[c]
        if int(s.notna().sum()) >= 8:
            lo = s.quantile(WINSORIZE_Q_LOW)
            hi = s.quantile(WINSORIZE_Q_HIGH)
            work[c] = s.clip(lo, hi)

    model_df = work[[sample_col, target_col] + feature_cols].drop_duplicates().copy()

    X = model_df[feature_cols].copy()
    y = model_df[target_col].copy()
    return X, y, feature_cols

def drop_high_corr_features_train_only(X_train, thr=0.98):
    if X_train.shape[1] <= 1:
        return X_train.copy(), []
    tmp = X_train.copy().fillna(X_train.median(numeric_only=True))
    corr = tmp.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_cols = [c for c in upper.columns if any(upper[c] > thr)]
    Xtr2 = X_train.drop(columns=drop_cols, errors="ignore").copy()
    return Xtr2, drop_cols

def fit_preprocess(X_tr):
    imp = SimpleImputer(strategy="median")
    vf = VarianceThreshold(0.0)
    X_imp = imp.fit_transform(X_tr)
    X_f = vf.fit_transform(X_imp)
    kept_mask = vf.get_support()
    kept_cols = [c for c, keep in zip(X_tr.columns.tolist(), kept_mask) if keep]
    return imp, vf, kept_cols

def transform_preprocess(imp, vf, X):
    return vf.transform(imp.transform(X))


# =========================
# MODEL FUNCTIONS
# =========================
def get_xgb_params(seed):
    return dict(
        booster="gbtree",
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        seed=seed,
        nthread=NTHREAD,
        eta=float(BEST_PARAMS["eta"]),
        max_depth=int(BEST_PARAMS["max_depth"]),
        min_child_weight=float(BEST_PARAMS["min_child_weight"]),
        subsample=float(BEST_PARAMS["subsample"]),
        colsample_bytree=float(BEST_PARAMS["colsample_bytree"]),
        gamma=float(BEST_PARAMS["gamma"]),
        reg_alpha=float(BEST_PARAMS["alpha"]),
        reg_lambda=float(BEST_PARAMS["lambda_"]),
        max_delta_step=float(BEST_PARAMS["max_delta_step"]),
    )

def train_final_model(X, y):
    # 高相关过滤仅用训练集决定
    X2, dropped_corr = drop_high_corr_features_train_only(X, CORR_THRESHOLD)

    imp, vf, kept_cols = fit_preprocess(X2)
    Xf = transform_preprocess(imp, vf, X2)

    z = np.log1p(np.asarray(y, float))
    dtrain = xgb.DMatrix(Xf, label=z)

    bst = xgb.train(
        get_xgb_params(RANDOM_STATE),
        dtrain,
        num_boost_round=NUM_BOOST_ROUND_FINAL,
        verbose_eval=False
    )

    return {
        "model": bst,
        "imputer": imp,
        "var_filter": vf,
        "kept_cols": kept_cols,
        "dropped_corr": dropped_corr,
    }

def predict_log1p_with_model(model_obj, X_pred):
    X2 = X_pred.drop(columns=model_obj["dropped_corr"], errors="ignore").copy()

    kept_cols = model_obj["kept_cols"]
    for c in kept_cols:
        if c not in X2.columns:
            X2[c] = np.nan
    X2 = X2[kept_cols]

    Xf = transform_preprocess(model_obj["imputer"], model_obj["var_filter"], X2)
    z_pred = model_obj["model"].predict(xgb.DMatrix(Xf))
    return z_pred


# =========================
# CV RESIDUAL FOR MC
# =========================
def estimate_log_residual_sigma(X, y):
    """
    用 repeated CV 的 OOF 残差估计 log1p 空间的残差标准差，
    同时收集 OOF 真值/预测值用于计算 CV 指标 (R2/log_R2/MAE/RMSE)
    """
    cv = RepeatedKFold(n_splits=CV_SPLITS, n_repeats=CV_REPEATS, random_state=RANDOM_STATE)
    residuals = []
    y_true_oof = []
    y_pred_oof = []

    for fid, (tr, va) in enumerate(cv.split(X, y), 1):
        X_tr = X.iloc[tr].copy()
        X_va = X.iloc[va].copy()
        y_tr = y.iloc[tr].copy()
        y_va = y.iloc[va].copy()

        X_tr2, dropped_corr = drop_high_corr_features_train_only(X_tr, CORR_THRESHOLD)
        X_va2 = X_va.drop(columns=dropped_corr, errors="ignore").copy()

        imp, vf, kept_cols = fit_preprocess(X_tr2)
        Xtr_f = transform_preprocess(imp, vf, X_tr2)

        for c in kept_cols:
            if c not in X_va2.columns:
                X_va2[c] = np.nan
        X_va2 = X_va2[kept_cols]
        Xva_f = transform_preprocess(imp, vf, X_va2)

        dtr = xgb.DMatrix(Xtr_f, label=np.log1p(np.asarray(y_tr, float)))
        bst = xgb.train(
            get_xgb_params(RANDOM_STATE + fid),
            dtr,
            num_boost_round=NUM_BOOST_ROUND_FINAL,
            verbose_eval=False
        )

        z_pred = bst.predict(xgb.DMatrix(Xva_f))
        res = np.log1p(np.asarray(y_va, float)) - z_pred
        residuals.append(res)

        y_pred_oof.append(np.clip(np.expm1(z_pred), 0, None))
        y_true_oof.append(np.asarray(y_va, float))

        del bst
        gc.collect()

    residuals = np.concatenate(residuals)
    y_true_oof = np.concatenate(y_true_oof)
    y_pred_oof = np.concatenate(y_pred_oof)
    sigma = float(np.std(residuals, ddof=1))

    cv_metrics = metric_pack(y_true_oof, y_pred_oof)
    return sigma, cv_metrics


# =========================
# HOLDOUT EVAL (random, stratified, bagging)
# =========================
def holdout_eval_random(X, y, bag_seeds=None):
    """
    随机留一 (20%) + 多 seed bagging 评估，复刻建模+SHAP脚本口径
    """
    if bag_seeds is None:
        bag_seeds = HOLDOUT_BAG_SEEDS

    bins = make_y_bins(y, 5)
    idx = np.arange(len(y))
    tr_idx, te_idx = train_test_split(
        idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=bins
    )

    X_tr_full = X.iloc[tr_idx].copy()
    y_tr_full = y.iloc[tr_idx].copy()
    X_te = X.iloc[te_idx].copy()
    y_te = y.iloc[te_idx].copy()

    preds_tr, preds_te = [], []
    for s in bag_seeds:
        Xtr2, dropped_corr = drop_high_corr_features_train_only(X_tr_full, CORR_THRESHOLD)
        Xte2 = X_te.drop(columns=dropped_corr, errors="ignore").copy()

        imp, vf, kept_cols = fit_preprocess(Xtr2)
        for c in kept_cols:
            if c not in Xtr2.columns:
                Xtr2[c] = np.nan
            if c not in Xte2.columns:
                Xte2[c] = np.nan
        Xtr2 = Xtr2[kept_cols]
        Xte2 = Xte2[kept_cols]

        Xtr_f = transform_preprocess(imp, vf, Xtr2)
        Xte_f = transform_preprocess(imp, vf, Xte2)

        bst = xgb.train(
            get_xgb_params(s),
            xgb.DMatrix(Xtr_f, label=np.log1p(np.asarray(y_tr_full, float))),
            num_boost_round=NUM_BOOST_ROUND_FINAL,
            verbose_eval=False
        )
        preds_tr.append(np.clip(np.expm1(bst.predict(xgb.DMatrix(Xtr_f))), 0, None))
        preds_te.append(np.clip(np.expm1(bst.predict(xgb.DMatrix(Xte_f))), 0, None))
        del bst
        gc.collect()

    y_tr_pred = np.mean(np.vstack(preds_tr), axis=0)
    y_te_pred = np.mean(np.vstack(preds_te), axis=0)

    return {
        "train_metrics": metric_pack(np.asarray(y_tr_full, float), y_tr_pred),
        "test_metrics": metric_pack(np.asarray(y_te, float), y_te_pred),
    }


# =========================
# PREDICTION DATA PREP
# =========================
def prepare_prediction_data(df, feature_cols_train):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    # 兼容 2021 文件里的 'NO2 ' 尾空格
    if "NO2" not in df.columns and "NO2 " in df.columns:
        df["NO2"] = df["NO2 "]

    cols_lower_map = {str(c).strip().lower(): c for c in df.columns}
    if "sample_name" not in cols_lower_map:
        raise ValueError(f"预测文件没有找到 sample_name 列，当前列名：{df.columns.tolist()}")
    sample_col = cols_lower_map["sample_name"]

    # 如果有 concentration 也保留，但预测时不用
    use_cols = [c for c in df.columns if c.lower() != "concentration"]

    work = df[use_cols].copy()
    work[sample_col] = work[sample_col].astype(str).str.strip()

    for c in work.columns:
        if c != sample_col:
            work[c] = safe_num_series(work[c])

    work = add_quarter_features(work)
    work = add_interaction_features(work)

    # 保证和训练特征一致
    for c in feature_cols_train:
        if c not in work.columns:
            work[c] = np.nan

    X_pred = work[feature_cols_train].copy()
    return work, sample_col, X_pred


# =========================
# MC OUTPUT / MONTHLY QUANTILES
# =========================
def mc_summary_from_log_pred(z_pred, sigma, n_mc=1000, seed=42):
    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0.0, scale=sigma, size=(len(z_pred), n_mc))
    z_mc = z_pred[:, None] + noise
    y_mc = np.expm1(z_mc)
    y_mc = np.clip(y_mc, 0, None)

    out = pd.DataFrame({
        "y_pred_log1p": z_pred,
        "y_pred_raw": np.expm1(z_pred),
        "y_pred_raw_mc_mean": y_mc.mean(axis=1),
        "y_pred_raw_mc_p025": np.quantile(y_mc, 0.025, axis=1),
        "y_pred_raw_mc_p25": np.quantile(y_mc, 0.25, axis=1),
        "y_pred_raw_mc_p50": np.quantile(y_mc, 0.50, axis=1),
        "y_pred_raw_mc_p75": np.quantile(y_mc, 0.75, axis=1),
        "y_pred_raw_mc_p975": np.quantile(y_mc, 0.975, axis=1),
    })
    out["y_pred_raw_mc_ci95_width"] = out["y_pred_raw_mc_p975"] - out["y_pred_raw_mc_p025"]
    out["y_pred_raw_mc_iqr"] = out["y_pred_raw_mc_p75"] - out["y_pred_raw_mc_p25"]
    return out

def add_month_index(df_pred):
    """
    2016行 = 168城 × 12月
    行顺序为每个城市连续12个月:
    1-12 = 城市1的1-12月
    13-24 = 城市2的1-12月
    ...
    所以 month = row_index % 12 + 1
    """
    out = df_pred.copy().reset_index(drop=True)
    out["month"] = (out.index % 12) + 1
    out["city_seq"] = (out.index // 12) + 1
    return out

def monthly_quantile_table(df_pred, year, value_col="y_pred_raw_mc_p50"):
    g = df_pred.groupby("month")[value_col]
    qdf = pd.DataFrame({
        "year": year,
        "month": list(range(1, 13)),
        "p10": g.quantile(0.10).values,
        "p25": g.quantile(0.25).values,
        "p50": g.quantile(0.50).values,
        "p75": g.quantile(0.75).values,
        "p90": g.quantile(0.90).values,
        "mean": g.mean().values,
        "n_cities": g.size().values,
    })
    return qdf


# =========================
# MAIN
# =========================
def main():
    ensure_outdir(OUT_DIR)

    train_fp = resolve_path(TRAIN_FILE_WINDOWS, TRAIN_FILE_FALLBACK)
    train_df = read_csv_auto(train_fp)

    print("读取训练文件:", train_fp)
    X_train, y_train, feature_cols_train = prepare_training_data(train_df)
    print("训练样本数:", len(y_train))
    print("训练特征数:", len(feature_cols_train))

    model_obj = train_final_model(X_train, y_train)
    print("最终入模特征数:", len(model_obj["kept_cols"]))

    print("\n估计 log1p 空间残差不确定性...")
    sigma_log, cv_metrics = estimate_log_residual_sigma(X_train, y_train)
    print("estimated sigma_log =", round(sigma_log, 6))

    print("\n计算建模性能指标 (R2 / log_R2 / MAE / RMSE) ...")
    hold = holdout_eval_random(X_train, y_train)
    metrics_df = pd.DataFrame([
        {"evaluation": "RepeatedCV_OOF", **cv_metrics},
        {"evaluation": "RandomHoldout_train", **hold["train_metrics"]},
        {"evaluation": "RandomHoldout_test", **hold["test_metrics"]},
    ])
    metrics_file = os.path.join(OUT_DIR, "model_performance_metrics.csv")
    metrics_df.to_csv(metrics_file, index=False, encoding="utf-8-sig")

    print("\n===== 模型性能 (随机CV / 留一holdout) =====")
    for _, r in metrics_df.iterrows():
        print("  {evaluation:20s} R2={R2:.4f}  log_R2={log_R2:.4f}  MAE={MAE:.3f}  RMSE={RMSE:.3f}".format(**r))
    print("  已保存:", metrics_file)

    all_quantiles = []

    for year in YEAR_ORDER:
        pred_fp = resolve_path(PRED_FILES[year], PRED_FILES_FALLBACK[year])
        pred_df = read_csv_auto(pred_fp)

        print(f"\n读取预测文件 {year}: {pred_fp} shape={pred_df.shape}")

        work_pred, sample_col, X_pred = prepare_prediction_data(pred_df, feature_cols_train)
        z_pred = predict_log1p_with_model(model_obj, X_pred)

        mc_df = mc_summary_from_log_pred(
            z_pred=z_pred,
            sigma=sigma_log,
            n_mc=MC_SAMPLES,
            seed=RANDOM_STATE + int(year)
        )

        # 仿照气相示例格式，第一列用 city 作为顺序ID
        out_df = pd.DataFrame({
            "city": np.arange(1, len(mc_df) + 1)
        })
        out_df = pd.concat([out_df, mc_df], axis=1)
        out_df = add_month_index(out_df)

        # 保存逐行 MC 结果（仿 predictions_YYYY_monthly_with_ci_mc.csv）
        out_file = os.path.join(OUT_DIR, f"predictions_{year}_monthly_with_ci_mc.csv")
        out_df.to_csv(out_file, index=False, encoding="utf-8-sig")

        # 月度分位数统计，基于 MC p50（更稳）
        qdf = monthly_quantile_table(out_df, year)
        q_file = os.path.join(OUT_DIR, f"{year}_monthly_quantiles_from_mc_p50.csv")
        qdf.to_csv(q_file, index=False, encoding="utf-8-sig")

        # 同时输出逐行点预测结果（含 pred_concentration 列，兼容原 2019/2021/2023 点预测版本）
        point_out = pred_df.copy()
        point_out.columns = [str(c).strip() for c in point_out.columns]
        point_out["pred_concentration"] = mc_df["y_pred_raw"].values
        point_out = add_month_index(point_out)
        point_file = os.path.join(OUT_DIR, f"{year}_逐行预测结果.csv")
        point_out.to_csv(point_file, index=False, encoding="utf-8-sig")

        all_quantiles.append(qdf)

        print(f"{year} 输出完成：")
        print("  ", out_file)
        print("  ", q_file)
        print("  ", point_file)

    # 合并总表
    all_q = pd.concat(all_quantiles, axis=0, ignore_index=True)
    all_q_file = os.path.join(OUT_DIR, "2015_2017_2019_2021_2023_monthly_quantiles_from_mc_p50.csv")
    all_q.to_csv(all_q_file, index=False, encoding="utf-8-sig")

    # 保存模型信息
    info = {
        "train_file": train_fp,
        "prediction_files": {
            y: resolve_path(PRED_FILES[y], PRED_FILES_FALLBACK[y]) for y in YEAR_ORDER
        },
        "n_train_samples": int(len(y_train)),
        "n_train_features_raw": int(len(feature_cols_train)),
        "n_train_features_final": int(len(model_obj["kept_cols"])),
        "best_params": BEST_PARAMS,
        "corr_dropped_features": model_obj["dropped_corr"],
        "sigma_log": sigma_log,
        "mc_samples": MC_SAMPLES,
        "metrics_repeated_cv_oof": cv_metrics,
        "metrics_random_holdout_train": hold["train_metrics"],
        "metrics_random_holdout_test": hold["test_metrics"],
    }
    with open(os.path.join(OUT_DIR, "预测说明.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)

    print("\n全部完成。结果目录：", OUT_DIR)
    print("总汇文件：", all_q_file)


if __name__ == "__main__":
    main()


读取训练文件: E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\20260311颗粒相ClPAHs建模-3.csv
训练样本数: 328
训练特征数: 34
最终入模特征数: 33

估计 log1p 空间残差不确定性...
estimated sigma_log = 0.533486

计算建模性能指标 (R2 / log_R2 / MAE / RMSE) ...

===== 模型性能 (随机CV / 留一holdout) =====
  RepeatedCV_OOF       R2=0.5645  log_R2=0.7579  MAE=11.967  RMSE=30.787
  RandomHoldout_train  R2=0.7811  log_R2=0.9644  MAE=6.020  RMSE=23.223
  RandomHoldout_test   R2=0.4910  log_R2=0.6947  MAE=11.337  RMSE=22.760
  已保存: C:\Users\HAUWEI\XGBoost建模\颗粒相ClPAHs_全年份预测_MC\model_performance_metrics.csv

读取预测文件 2015: E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2015年预测.csv shape=(2016, 26)
2015 输出完成：
   C:\Users\HAUWEI\XGBoost建模\颗粒相ClPAHs_全年份预测_MC\predictions_2015_monthly_with_ci_mc.csv
   C:\Users\HAUWEI\XGBoost建模\颗粒相ClPAHs_全年份预测_MC\2015_monthly_quantiles_from_mc_p50.csv
   C:\Users\HAUWEI\XGBoost建模\颗粒相ClPAHs_全年份预测_MC\2015_逐行预测结果.csv

读取预测文件 2017: E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2017年预测.csv shape=(2016, 26)
2